# VM End-To-End Setup

Notebook nay dung de setup lai mot GPU VM tu dau va chay full end-to-end tren VM:

```text
Expo app -> Cloudflare backend tunnel -> FastAPI backend on VM
  -> local bbox cleanup
  -> Hunyuan worker on same VM
  -> GLB result
```

Khong dung Gemini/Nano Banana. Khong dung TripoSR. Backend tren VM se goi worker noi bo qua `http://127.0.0.1:8010`.

## Start here if the VM was stopped

A stopped VM kills JupyterLab and all quick Cloudflare tunnels. Before running this notebook, reopen Jupyter from SSH.


### A. Start JupyterLab in tmux

Run in SSH tab 1:

```bash
tmux ls || true
tmux new -s jupyter
cd ~/work
source venv/bin/activate 2>/dev/null || true
source ~/.bashrc
jupyter lab --no-browser --ip=127.0.0.1 --port=8888 --ServerApp.allow_remote_access=True
```

If `~/work/venv` does not exist yet on a fresh VM, create a minimal Jupyter venv first:

```bash
sudo apt-get update
sudo apt-get install -y python3-venv python3-pip tmux curl wget git
mkdir -p ~/work
cd ~/work
python3 -m venv venv
source venv/bin/activate
python -m pip install -U pip wheel "setuptools<82" jupyterlab
jupyter lab --no-browser --ip=127.0.0.1 --port=8888 --ServerApp.allow_remote_access=True
```

If tmux says `duplicate session: jupyter`, attach instead:

```bash
tmux attach -t jupyter
```

Copy the Jupyter token. Detach without stopping Jupyter with `Ctrl+B`, then `D`.


### B. Expose Jupyter through Cloudflare

Run in SSH tab 2:

```bash
if ! command -v cloudflared >/dev/null 2>&1; then
  curl -L -o /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
  sudo dpkg -i /tmp/cloudflared.deb || sudo apt-get install -f -y
fi
cloudflared tunnel --url http://127.0.0.1:8888
```

Open on Windows/Chrome:

```text
https://YOUR_JUPYTER_TUNNEL.trycloudflare.com/lab?token=YOUR_JUPYTER_TOKEN
```

Then open this notebook from:

```text
~/work/AI_3D_Reconstruction_Systerm/deploy/VM_END_TO_END_SETUP.ipynb
```


## 0. Config

Sua `REPO_REF` neu ban da merge branch nay vao main.

In [ ]:
REPO_URL = "https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git"
REPO_REF = "codex/hunyuan-shape-then-paint"
WORK_DIR = "$HOME/work"
REPO_DIR = "$HOME/work/AI_3D_Reconstruction_Systerm"
HUNYUAN_DIR = "$HOME/work/Hunyuan3D-2"
print(REPO_URL, REPO_REF)

## 1. Check VM GPU

In [ ]:
!nvidia-smi
!python3 --version
!free -h
!df -h | head -20

## 2. OS, NVIDIA driver, and CUDA Toolkit

Recommended VM:

```text
OS: Debian 12 or Ubuntu 22.04/24.04
GPU: NVIDIA L4/T4 or better
RAM: 30GB minimum, 50GB+ preferred
Disk: 100GB minimum, 150GB+ preferred
```

If `nvidia-smi` and `nvcc --version` already work, skip to section 3. If this is a fresh Debian 12 VM, run the install cells below. Driver install requires reboot.

In [ ]:
%%bash
set -euo pipefail
echo 'OS:'
cat /etc/os-release | sed -n '1,6p'
echo
echo 'GPU PCI:'
lspci | grep -i nvidia || true
echo
echo 'nvidia-smi:'
nvidia-smi || true
echo
echo 'nvcc:'
nvcc --version || true

### 2A. Debian 12 driver install

Run this only if `nvidia-smi` is missing/broken. After this cell finishes, reboot the VM, reconnect SSH/Jupyter, then continue from section 2B.

In [ ]:
%%bash
set -euo pipefail
if nvidia-smi >/dev/null 2>&1; then
  echo 'NVIDIA driver already works; skipping driver install.'
  exit 0
fi
if [ -f /etc/debian_version ]; then
  sudo sed -i 's/^Components: main$/Components: main contrib non-free non-free-firmware/' /etc/apt/sources.list.d/debian.sources || true
  sudo apt-get update
  sudo apt-get install -y --no-install-recommends linux-headers-$(uname -r) dkms build-essential nvidia-driver firmware-misc-nonfree
  echo 'Driver installed. Reboot now, then rerun nvidia-smi before continuing.'
  echo 'Command: sudo reboot'
else
  echo 'This driver cell is written for Debian. For Ubuntu, use Google Cloud GPU driver install docs or an image with NVIDIA drivers preinstalled.'
fi

### 2B. CUDA Toolkit / nvcc install

Texture extension build requires CUDA Toolkit (`nvcc`), not only the NVIDIA driver. Prefer CUDA 12.x because PyTorch is installed from CUDA 12.6 wheels.

In [ ]:
%%bash
set -euo pipefail
if nvcc --version >/dev/null 2>&1; then
  echo 'nvcc already works; skipping CUDA toolkit install.'
  nvcc --version
  exit 0
fi
if [ -f /etc/debian_version ]; then
  cd /tmp
  wget -nc https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64/cuda-keyring_1.1-1_all.deb
  sudo dpkg -i cuda-keyring_1.1-1_all.deb
  sudo apt-get update
  sudo apt-get install -y cuda-toolkit-12-6 || sudo apt-get install -y cuda-toolkit-12-8
  grep -qxF 'export CUDA_HOME=/usr/local/cuda' ~/.bashrc || echo 'export CUDA_HOME=/usr/local/cuda' >> ~/.bashrc
  grep -qxF 'export PATH=$CUDA_HOME/bin:$PATH' ~/.bashrc || echo 'export PATH=$CUDA_HOME/bin:$PATH' >> ~/.bashrc
  grep -qxF 'export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH' ~/.bashrc || echo 'export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH' >> ~/.bashrc
  export CUDA_HOME=/usr/local/cuda
  export PATH=$CUDA_HOME/bin:$PATH
  export LD_LIBRARY_PATH=$CUDA_HOME/lib64:${LD_LIBRARY_PATH:-}
  nvcc --version
else
  echo 'This CUDA toolkit cell is written for Debian. Install a CUDA 12.x toolkit for your OS, then continue.'
fi

## 3. Install base tools and cloudflared

In [ ]:
%%bash
set -euo pipefail
sudo apt-get update
sudo apt-get install -y --no-install-recommends git curl wget tmux build-essential python3-pip python3-venv ninja-build ca-certificates
if ! command -v cloudflared >/dev/null 2>&1; then
  curl -L -o /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
  sudo dpkg -i /tmp/cloudflared.deb || sudo apt-get install -f -y
fi
cloudflared --version

## 4. Clone repo branch

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git"
REPO_REF="codex/hunyuan-shape-then-paint"
WORK_DIR="$HOME/work"
REPO_DIR="$WORK_DIR/AI_3D_Reconstruction_Systerm"
mkdir -p "$WORK_DIR"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --branch "$REPO_REF" "$REPO_URL" "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin "$REPO_REF"
  git -C "$REPO_DIR" checkout "$REPO_REF"
  git -C "$REPO_DIR" pull --ff-only
fi
git -C "$REPO_DIR" status --short
git -C "$REPO_DIR" rev-parse --short HEAD

## 5. Setup Hunyuan worker service

Cell nay cai Hunyuan3D-2 vao `$HOME/work/venv`, copy worker FastAPI, va tao service `hunyuan-worker` port `8010`.

In [ ]:
%%bash
set -euo pipefail
cd "$HOME/work/AI_3D_Reconstruction_Systerm"
export REPO_REF="codex/hunyuan-shape-then-paint"
bash scripts/gcp_hunyuan_worker_bootstrap.sh

## Verify pinned Hunyuan runtime versions

The bootstrap script fails if these versions drift, but this cell prints them again for audit/debug.


In [ ]:
%%bash
set -euo pipefail
source ~/work/venv/bin/activate
python - <<'PY'
import importlib.metadata as metadata
import torch
expected = {
    'diffusers': '0.31.0',
    'transformers': '4.46.3',
    'tokenizers': '0.20.3',
    'huggingface_hub': '0.26.2',
    'accelerate': '1.1.1',
}
print('torch==', torch.__version__)
print('cuda_available=', torch.cuda.is_available())
print('gpu=', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA')
if '+cu' not in torch.__version__:
    raise SystemExit(f'Expected CUDA torch wheel, got {torch.__version__}')
for package, version in expected.items():
    actual = metadata.version(package)
    print(f'{package}=={actual}')
    if actual != version:
        raise SystemExit(f'Expected {package}=={version}, got {actual}')
print('version pins OK')
PY


## 6. Verify worker

In [ ]:
%%bash
set -euo pipefail
sudo systemctl status hunyuan-worker --no-pager | head -40
echo "Waiting for Hunyuan worker health..."
for attempt in $(seq 1 60); do
  body=$(curl -fsS --max-time 5 http://127.0.0.1:8010/health 2>/tmp/hunyuan_worker_health.err || true)
  if [ -n "$body" ] && echo "$body" | python3 -m json.tool; then
    echo "Hunyuan worker health OK after ${attempt}s."
    exit 0
  fi
  if ! sudo systemctl is-active --quiet hunyuan-worker; then
    echo "hunyuan-worker stopped while waiting for health. Recent logs:"
    sudo journalctl -u hunyuan-worker -n 120 --no-pager
    exit 1
  fi
  sleep 1
done
echo "Hunyuan worker did not return valid JSON health within 60 seconds. Recent logs:"
sudo journalctl -u hunyuan-worker -n 120 --no-pager
exit 1

## 7. Setup FastAPI backend on the same VM

Backend VM se goi Hunyuan worker noi bo qua `http://127.0.0.1:8010`, nen khong can worker tunnel rieng.

In [ ]:
%%bash
set -euo pipefail
cd "$HOME/work/AI_3D_Reconstruction_Systerm"
export HUNYUAN_REMOTE_URL="http://127.0.0.1:8010"
export MPLBACKEND=Agg
bash scripts/gcp_backend_vm_bootstrap.sh

## 8. Verify backend

In [ ]:
%%bash
set -euo pipefail
sudo systemctl status ai-3d-backend --no-pager | head -50
echo "Waiting for backend health..."
for attempt in $(seq 1 60); do
  body=$(curl -fsS --max-time 5 http://127.0.0.1:8000/health 2>/tmp/ai_3d_backend_health.err || true)
  if [ -n "$body" ] && echo "$body" | python3 -m json.tool; then
    echo "Backend health OK after ${attempt}s."
    exit 0
  fi
  if ! sudo systemctl is-active --quiet ai-3d-backend; then
    echo "ai-3d-backend stopped while waiting for health. Recent logs:"
    sudo journalctl -u ai-3d-backend -n 120 --no-pager
    exit 1
  fi
  sleep 1
done
echo "Backend did not return valid JSON health within 60 seconds. Recent logs:"
sudo journalctl -u ai-3d-backend -n 120 --no-pager
exit 1

## 9. Start Cloudflare tunnel for backend

Expo se goi URL nay. Copy URL `https://....trycloudflare.com` tu output cell.

In [ ]:
%%bash
set -euo pipefail
cd "$HOME/work/AI_3D_Reconstruction_Systerm"
SESSION=backend-tunnel TARGET_URL=http://127.0.0.1:8000 bash deploy/scripts/start_tunnel_tmux.sh
sleep 8
tmux capture-pane -t backend-tunnel -p -S -120 | tee /tmp/backend_tunnel.log
grep -o 'https://[^ ]*trycloudflare.com' /tmp/backend_tunnel.log | tail -1 || true

## 10. Smoke test backend tunnel

Paste backend tunnel URL vao bien duoi roi chay cell.

In [ ]:
BACKEND_TUNNEL_URL = "https://YOUR_BACKEND_TUNNEL.trycloudflare.com"  # replace this
print(BACKEND_TUNNEL_URL)

In [ ]:
import json, urllib.request
with urllib.request.urlopen(BACKEND_TUNNEL_URL.rstrip('/') + '/health', timeout=30) as r:
    print(json.dumps(json.load(r), indent=2)[:2000])

## 11. Local clean smoke test

Test preprocess endpoint truoc khi gui job Hunyuan.

In [ ]:
%%bash
set -euo pipefail
curl -s -X POST "http://127.0.0.1:8000/preprocess/clean-image" \
  -F "image=@$HOME/work/AI_3D_Reconstruction_Systerm/project/samples/chair_demo.png" \
  -F "bbox_x=10" \
  -F "bbox_y=10" \
  -F "bbox_width=400" \
  -F "bbox_height=400" \
  -F "job_id=vm-clean-smoke" | python3 -m json.tool | head -80

## 12. End-to-end reconstruct smoke test

Cell nay tao job Hunyuan that, co the mat vai phut. Neu worker dang busy, doi job truoc xong.

In [ ]:
%%bash
set -euo pipefail
RESP=$(curl -s -X POST "http://127.0.0.1:8000/reconstruct-bbox" \
  -F "image=@$HOME/work/AI_3D_Reconstruction_Systerm/project/samples/chair_demo.png" \
  -F "bbox_x=10" \
  -F "bbox_y=10" \
  -F "bbox_width=400" \
  -F "bbox_height=400")
echo "$RESP" | python3 -m json.tool
JOB_ID=$(printf '%s' "$RESP" | python3 -c "import sys,json; print(json.load(sys.stdin)['job_id'])")
echo "JOB_ID=$JOB_ID"
for i in $(seq 1 120); do
  STATUS=$(curl -s "http://127.0.0.1:8000/reconstruction-jobs/$JOB_ID")
  echo "$STATUS" | python3 -m json.tool | head -40
  echo "$STATUS" | grep -q '"status": "done"' && break
  echo "$STATUS" | grep -q '"status": "failed"' && exit 1
  sleep 5
done

## 13. Expo command on laptop

Open a new PowerShell terminal on Windows/laptop. Use the backend tunnel URL from step 9, not the Jupyter tunnel URL:

```powershell
cd C:\Users\pminh\Desktop\MyProject\AI_3D_Reconstruction_Systerm_TangDien02\mobile
$env:EXPO_PUBLIC_API_BASE_URL="https://YOUR_BACKEND_TUNNEL.trycloudflare.com"
npm start -- --host lan
```

If Expo still uses an old backend URL, clear Metro cache:

```powershell
npm start -- --host lan -c
```

If backend runs on Windows instead of the VM, use the Windows LAN IP:

```powershell
$env:EXPO_PUBLIC_API_BASE_URL="http://192.168.1.5:8000"
npm start -- --host lan
```

## 14. Logs

In [ ]:
# Worker logs
!sudo journalctl -u hunyuan-worker -n 120 --no-pager

In [ ]:
# Backend logs
!sudo journalctl -u ai-3d-backend -n 120 --no-pager

In [ ]:
# Backend tunnel logs
!tmux capture-pane -t backend-tunnel -p -S -120

## 15. Restart commands

```bash
sudo systemctl restart hunyuan-worker
sudo systemctl restart ai-3d-backend
tmux kill-session -t backend-tunnel
SESSION=backend-tunnel TARGET_URL=http://127.0.0.1:8000 bash ~/work/AI_3D_Reconstruction_Systerm/deploy/scripts/start_tunnel_tmux.sh
```